In [1]:
import pathlib
import os
import re

from IPython.display import display, Markdown
import polars

from utils import list_code

EVALUATION_DIR = pathlib.Path.cwd()
INPUT_PATH = EVALUATION_DIR / "output_dataset"

os.environ["INPUT_PATH"] = str(INPUT_PATH)

# Extract package data from PCAPs

TBD:
- **Parallel needs to be installed for this!**
- wireshark preferences:
  ```yaml
  dtls.psk: f28aa24f8cb8a7cc483cf336c0be70047f29
  ```
  wireshark oscore_contexts:
  ```csv
  "01","64617461","f28aa24f8cb8a7cc483cf336c0be70047f29","","","AES-CCM-16-64-128 (CCM*)"
  "64617461","01","f28aa24f8cb8a7cc483cf336c0be70047f29","","","AES-CCM-16-64-128 (CCM*)"
  "02","646e73","0e08f702e6508657e3a0fd996a042531fc89","","","AES-CCM-16-64-128 (CCM*)"
  "646e73","02","0e08f702e6508657e3a0fd996a042531fc89","","","AES-CCM-16-64-128 (CCM*)"
  "03","70726f7879","b35c9719b3594553315a210b18720da0e750","","","AES-CCM-16-64-128 (CCM*)"
  "70726f7879","03","b35c9719b3594553315a210b18720da0e750","","","AES-CCM-16-64-128 (CCM*)"
  ```

In [2]:
list_code(EVALUATION_DIR / "extract_from_pcaps.sh")

#!/usr/bin/env bash
#
# extract_from_pcaps.sh
# Copyright (C) 2025 TU Dresden
#
# Distributed under terms of the MIT license.
#

SCRIPT_DIR=$( cd -- "$( dirname -- "$(realpath "$0")" )" &> /dev/null && pwd )
PROCS=$(grep -c '^processor' /proc/cpuinfo)
if [ $PROCS -gt 64 ]; then
    # leave some resources to collegues ;-)
    PROCS=$(( (PROCS * 3) / 4))
fi
OUTPUT_DATASET="${OUTPUT_DATASET:-${SCRIPT_DIR}/output_dataset}"


extract_from_pcap() {
    PCAP="$1"

    if ! echo "$PCAP" | grep -Eq ".*\.upstream.pcapng"; then
        "${SCRIPT_DIR}/extract_eth.sh" "${PCAP}" > "${PCAP%.pcapng}".eth.csv
    fi
    "${SCRIPT_DIR}/extract_coap.sh" "${PCAP}" > "${PCAP%.pcapng}".coap.csv
}

export -f extract_from_pcap
export SCRIPT_DIR

find "${OUTPUT_DATASET}" -name "*.wpan.pcapng" -o -name "*.upstream.pcapng" |
    parallel -j"${PROCS}" extract_from_pcap

In [3]:
list_code(EVALUATION_DIR / "extract_eth.sh")

#! /bin/sh
#
# extract_eth.sh
# Copyright (C) 2025 TU Dresden
#
# Distributed under terms of the MIT license.
#

if [ $# -ne 1 ]; then
    echo "usage: $0 <pcap file>" >&2
    exit 1
fi

PCAP="$1"

FIELDS="frame.number frame.time_epoch eth.payload"
echo "${FIELDS}" | \
    awk 'BEGIN {OFS="\t"} { for (i = 1; i <= NF; i++) { printf "%s%s", $i, (i < NF) ? OFS : ORS } }'
tshark -Tfields -e frame.number -e frame.time_epoch -e data --disable-protocol ALL --enable-protocol eth -r "${PCAP}"

In [4]:
list_code(EVALUATION_DIR / "extract_coap.sh")

#! /bin/sh
#
# extract_eth.sh
# Copyright (C) 2025 TU Dresden
#
# Distributed under terms of the MIT license.
#

if [ $# -ne 1 ]; then
    echo "usage: $0 <pcap file>" >&2
    exit 1
fi

PCAP="$1"

FIELDS="frame.number frame.time_epoch frame.protocols"
FIELDS="${FIELDS} coap.code coap.request_first_in coap.mid coap.token coap.opt.ctype coap.opt.block_number coap.opt.block_mflag coap.opt.block_size"

if echo "$PCAP" | grep -q "oscore"; then
    FIELDS="${FIELDS} oscore.code oscore.opt.ctype oscore.opt.block_number oscore.opt.block_mflag oscore.opt.block_size"
fi


echo "${FIELDS}" | \
    awk 'BEGIN {OFS="\t"} { for (i = 1; i <= NF; i++) { printf "%s%s", $i, (i < NF) ? OFS : ORS } }'
tshark -Tfields $(for field in ${FIELDS}; do printf -- "-e %s " "${field}"; done) -r "${PCAP}"

In [5]:
!"{EVALUATION_DIR}"/extract_from_pcaps.sh

# Generate Training Data

In [6]:
COLUMN_ORDER = [
    "client.coap.response.recv_time_epoch",
    "client.type",
    "client.dns.qry.name",
    "client.dns.qry.type",
    "client.coap.url",
    "client.coap.media_type",
    "client.coap.response.code",
    "client.coap.response.payload",
    "frame.number",
    "frame.time_epoch",
    "frame.protocols",
    "eth.payload",
    "ipv6.prefix",
    "coap.is_response",
    "coap.code",
    "coap.request_first_in",
    "coap.mid",
    "coap.token",
    "coap.opt.ctype",
    "coap.opt.block_number",
    "coap.opt.block_mflag",
    "coap.opt.block_size",
    "oscore.code",
    "oscore.opt.ctype",
    "oscore.opt.block_number",
    "oscore.opt.block_mflag",
    "oscore.opt.block_size",
    "upstream.frame.number",
    "upstream.frame.time_epoch",
    "upstream.frame.protocols",
    "upstream.ipv6.prefix",
    "upstream.coap.code",
    "upstream.coap.request_first_in",
    "upstream.coap.mid",
    "upstream.coap.token",
    "upstream.coap.opt.ctype",
    "upstream.coap.opt.block_number",
    "upstream.coap.opt.block_mflag",
    "upstream.coap.opt.block_size",
    "upstream.oscore.code",
    "upstream.oscore.opt.ctype",
    "upstream.oscore.opt.block_number",
    "upstream.oscore.opt.block_mflag",
    "upstream.oscore.opt.block_size",
]

tables = {}

for root, dirs, files in INPUT_PATH.walk():
    for file in files:
        if match := re.search(
            r".*\.((wpan|upstream)\.(coap|eth)\.csv|(client|proxy)\.log)",
            file,
        ):
            scenario = file.split(".")[0]
            if scenario not in tables:
                tables[scenario] = {}
            if not (
                file.startswith("oscore")
                or file.startswith("coaps-p")
                or file.startswith("coap-p")
            ) and (
                match[1] == "upstream"
                or match[4] == "proxy"
            ):
                display(Markdown(f"## No upstream {scenario}"))
                continue
            elif match[1] == "client.log":
                table = "client"
            elif match[1] == "proxy.log":
                table = "proxy"
            else:
                table = match[3]
            if match[2] == "upstream":
                table = f"{table}_upstream"
            tables[scenario][table] = polars.read_csv(
                root / file,
                separator="\t"
            )


for scenario in tables:
    assert all(table in tables[scenario] for table in ["client", "eth", "coap"])
    assert "coap_upstream" not in tables[scenario] or "proxy" in tables[scenario]
    df = tables[scenario]["eth"].join(
        tables[scenario]["coap"][
            [
                col for col in tables[scenario]["coap"].columns
                if col != "frame.time_epoch"
            ]
        ],
        on="frame.number",
        how="inner"
    )
    if "coap_upstream" in tables[scenario]:
        df = df.with_columns(((df["coap.code"] & (7 << 5)) != 0).alias("coap.is_response"))
        upstream_df = tables[scenario]["coap_upstream"]
        upstream_df = upstream_df.with_columns(((upstream_df["coap.code"] & (7 << 5)) != 0).alias("coap.is_response"))
        df = df.join(
            tables[scenario]["proxy"],
            left_on="coap.token",
            right_on="old_token",
            how="left",
        )
        df = df.join(
            upstream_df,
            left_on=["coap.is_response", "new_token"],
            right_on=["coap.is_response", "coap.token"],
            how="left",
            suffix=".upstream",
        )
        df = df.rename({"new_token": "coap.token.upstream"}).join(
            tables[scenario]["client"],
            left_on="coap.token.upstream",
            right_on="token",
        ).rename(
            lambda c: f"upstream.{c[:-9]}" if c.endswith(".upstream") else c
        ).rename(
            {
                "timestamp": "client.coap.response.recv_time_epoch",
                "wpan_prefix": "ipv6.prefix",
                "upstream_prefix": "upstream.ipv6.prefix",
                "type": "client.type",
                "query_name": "client.dns.qry.name",
                "query_type": "client.dns.qry.type",
                "url": "client.coap.url",
                "media_type": "client.coap.media_type",
                "response_code": "client.coap.response.code",
                "response_payload": "client.coap.response.payload",
            }
        )
    else:
        df = df.join(
            tables[scenario]["client"],
            left_on="coap.token",
            right_on="token",
        ).rename(
            {
                "timestamp": "client.coap.response.recv_time_epoch",
                "wpan_prefix": "ipv6.prefix",
                "upstream_prefix": "upstream.ipv6.prefix",
                "type": "client.type",
                "query_name": "client.dns.qry.name",
                "query_type": "client.dns.qry.type",
                "url": "client.coap.url",
                "media_type": "client.coap.media_type",
                "response_code": "client.coap.response.code",
                "response_payload": "client.coap.response.payload",
            }
        )
    assert all(c in COLUMN_ORDER for c in df.columns)
    df = df.select([c for c in COLUMN_ORDER if c in df.columns])
    max_len = df["eth.payload"].str.len_chars().max()
    if max_len is None:
        print(f"Skipping {scenario}")
        continue
    df.write_csv(INPUT_PATH / f"{scenario}.merged.csv", separator=";")
    df_training = df.select(
        ["frame.number", "frame.time_epoch", "client.type", "eth.payload"]
        # add upstream mid and token to ensure only duplicate frames
        # due to upstream retransmissions are deleted
        + (["upstream.coap.mid", "upstream.coap.token"] if "upstream.coap.mid" in df.columns else [])
    ).with_columns(
        **{
            "eth.payload": polars.col("eth.payload").str.pad_end(max_len, "x")
        }
    )
    del df
    # deduplicate frames that were duplicated due to upstream retransmission
    df_training.unique(keep="last", maintain_order=True).select(
        ["frame.number", "frame.time_epoch", "client.type", "eth.payload"]
    ).write_csv(
        INPUT_PATH / f"{scenario}.training.csv", separator=";"
    )
    del df_training

# Check If Files are Complete

In [7]:
for root, dirs, files in INPUT_PATH.walk():
    for file in files:
        match = re.search(r".*\.training\.csv$", file)
        if not match:
            continue
        df = polars.read_csv(
            root / file.replace("training.csv", "wpan.coap.csv"),
            separator="\t",
        )
        if file.startswith("coaps"):
            # DTLS control packages (e.g. handshake) should be excluded
            non_coap_frames = set(
                df.filter(
                    df["frame.protocols"].str.ends_with(":dtls")
                )["frame.number"].to_list()
            )
        else:
            non_coap_frames = set()
        # Empty CoAP ACKs should be excluded
        non_coap_frames |= set(
            df.filter(
                df["coap.code"] == 0
            )["frame.number"].to_list()
        )
        del df
        df = polars.read_csv(root / file, separator=";")
        exp_frames = polars.DataFrame(
            {"frame.exp": [i for i in range(1, df["frame.number"].max() + 1) if i not in non_coap_frames]}
        )
        if (df.is_empty() or df.shape[0] != exp_frames.shape[0]):
            display(Markdown(f"## ❌ Error in {file}"))
            display(Markdown(f"Number of Non-CoAP frames {len(non_coap_frames)}"))
            display(Markdown("### Broken table"))
            display(df)
            display(Markdown(f"### Duplicate frame numbers"))
            display(df.filter(polars.count("frame.number").over(df.columns) > 1))
            display(Markdown(f"### Expected CoAP frames [{df.shape[0] - len(non_coap_frames)}]"))
            display(exp_frames)
            display(Markdown(f"### Different frame numbers"))
            display(
                polars.DataFrame(
                    {
                        "diff": sorted(
                            set(df["frame.number"].to_list()).difference(
                                set(exp_frames["frame.exp"].to_list()),
                            ) | 
                            set(exp_frames["frame.exp"].to_list()).difference(
                                set(df["frame.number"].to_list()),
                            )
                        )
                    }
                )
            )
            display(Markdown("### Non-CoAP frames"))
            display(df.filter(df["frame.number"].is_in(non_coap_frames)))
            display(polars.DataFrame({"non_coap_frames": sorted(non_coap_frames)}))
            merged = file.replace("training.csv", "merged.csv")
            display(Markdown(f"### [{merged}]({(root / merged).relative_to(EVALUATION_DIR)})"))
            merged_df = polars.read_csv(root / merged, separator=";")
            display(merged_df)
            wpan_coap = file.replace("training.csv", "wpan.coap.csv")
            display(Markdown(f"### [{wpan_coap}]({(root / wpan_coap).relative_to(EVALUATION_DIR)})"))
            display(polars.read_csv(root / wpan_coap, separator="\t"))
            continue
        df = df.with_columns(exp_frames)
        check = df["frame.number"] != df["frame.exp"]
        if check.any():
            display(Markdown("### Different from expected"))
            stored = file.replace(".training.csv", ".training_broken.csv")
            display(Markdown(f"#### [{stored}]({(root / stored).relative_to(EVALUATION_DIR)})"))
            display(df.filter(check))
            df.write_csv(root / stored)
            break
        display(Markdown(f"## ✅ {file} OK"))
        del df

## ✅ coap-d1_json_dns_message.training.csv OK

## ✅ coap-d2_json_dns_message.training.csv OK

## ✅ coap-p1_json_dns_message.training.csv OK

## ✅ coap-p2_json_dns_message.training.csv OK

## ✅ coaps-d1_json_dns_cbor.training.csv OK

...

## ✅ oscore-base-p2_cbor_dns_message.training.csv OK

## ✅ oscore-base-d1_cbor_dns_cbor.training.csv OK

## ✅ oscore-base-p1_cbor_dns_cbor.training.csv OK

## ✅ oscore-base-d2_cbor_dns_cbor.training.csv OK

## ✅ oscore-base-p2_cbor_dns_cbor.training.csv OK

# Compress Training Data

In [8]:
%%bash

for csv in "${INPUT_PATH}"/*.training.csv; do
    pigz -9 -f "${csv}"
done
chmod +x "${INPUT_PATH}"/*.training.csv.gz